# **R-learner: DML for the effect function**

Practice for the module [“Heterogeneous effects”](https://ai-interpretability.school/learn/5/0/0), lesson “Metalearners: S, T, and X”, step “The R-learner: DML for a function”.

In the lesson, the R-learner ([Nie and Wager, 2021](https://arxiv.org/abs/1712.04912)) was a formula: subtract from the outcome and from the treatment their predictions from the features (the Robinson residuals), then find the effect function $\tau(x)$ by a weighted regression of the pseudo-outcome $\tilde Y_i/\tilde T_i$ with weights $\tilde T_i^2$. Here we build it by hand and check three claims of the lesson:

1. orthogonalization lets us fit $\tau$ with a simple, strongly regularized model;
2. the scheme breaks under strong group imbalance, because the weight $\tilde T_i^2$ of control objects is close to zero;
3. the R-learner also gives a criterion for choosing an effect model that does not require the true effect.

Plan of the notebook:

1. the IHDP data with a known true effect;
2. Robinson residuals with cross-fitting;
3. the R-learner as a weighted regression of the pseudo-outcome;
4. comparison with the S-, T-, X-, and DR-learner;
5. sensitivity to the regularization of the effect model;
6. where the scheme breaks: the propensity model and group imbalance;
7. the R-learner loss as a model selection criterion;
8. the library call `econml.dml.NonParamDML`.

The notebook runs on CPU; the whole run takes about a minute.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from econml.metalearners import SLearner, TLearner, XLearner
from econml.dr import DRLearner
from econml.dml import NonParamDML

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # spurious numpy warnings on some CPUs
SEED = 0
pd.set_option('display.width', 140)

## 1. The IHDP data

IHDP (Infant Health and Development Program) is a randomized experiment with premature low-birth-weight infants: the treatment is home visits by specialists and an educational child care center. [Hill (2011)](https://doi.org/10.1198/jcgs.2010.08162) turned it into a semi-synthetic dataset that became a standard benchmark for CATE estimates:

- the features are real: 25 characteristics of the child and the mother, 6 continuous (standardized) and 19 binary;
- a non-random part of the treated children was removed, so the groups are no longer comparable: 747 children remain, 139 of them treated;
- the outcomes are generated by a known formula (Hill's response surface B), so for every child both expected potential outcomes $\mu_0(x)$ and $\mu_1(x)$ and the true effect $\tau(x)=\mu_1(x)-\mu_0(x)$ are known.

We take the first of the ten replications published by the CEVAE authors ([Louizos et al., 2017](https://github.com/AMLab-Amsterdam/CEVAE)). 70% of the children go to training, 30% to a held-out check; the split is stratified by treatment.

We measure the error of an estimate the way Hill does: $\mathrm{PEHE}=\sqrt{\tfrac1n\sum_i(\hat\tau(x_i)-\tau(x_i))^2}$ on the held-out part. Real data have no such metric; in section 7 we look at what can replace it.

In [ ]:
URL = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/ihdp_npci_1.csv'
cols = ['treatment', 'y_factual', 'y_cfactual', 'mu0', 'mu1'] + [f'x{i}' for i in range(1, 26)]
df = pd.read_csv(URL, header=None, names=cols)

X = df[[f'x{i}' for i in range(1, 26)]].values
T = df.treatment.values.astype(int)
Y = df.y_factual.values
tau = (df.mu1 - df.mu0).values                 # true CATE: known only because the data are semi-synthetic

tr, te = train_test_split(np.arange(len(df)), test_size=0.3, random_state=SEED, stratify=T)
print('children:', len(df), '| treated:', T.sum(), f'({T.mean():.3f})')
print('training:', len(tr), '| held-out:', len(te))
print(f'true effect: mean {tau.mean():.2f}, standard deviation {tau.std():.2f}')


def pehe(est):
    # error of a CATE estimate on the held-out part
    return np.sqrt(np.mean((est - tau[te]) ** 2))


def gb(depth=3):
    # outcome model shared by all schemes of the notebook
    return GradientBoostingRegressor(n_estimators=200, max_depth=depth, learning_rate=0.05, random_state=SEED)

## 2. Robinson residuals with cross-fitting

The first step of the R-learner is two auxiliary models: a model of the mean outcome $\hat m(x)\approx\mathbb E[Y\mid X=x]$ on the whole sample, without splitting into groups, and a propensity model $\hat e(x)\approx\mathbb P(T=1\mid X=x)$. The residuals of an object are computed by models that did not see that object (cross-fitting): otherwise an overfitted $\hat m$ would absorb part of the effect together with the noise.

**Task 1. On the training part, compute the predictions `m_hat` and `e_hat` with `cross_val_predict` and the split `kf = KFold(5, shuffle=True, random_state=SEED)`: the outcome model is `gb()`, the propensity model is `LogisticRegression(max_iter=1000)` (probabilities via `method='predict_proba'`, column 1). Compute the residuals `y_res` $=\tilde Y$ and `t_res` $=\tilde T$ and print which share of the total weight $\sum_i\tilde T_i^2$ falls on the treated objects of the training part.**

In [ ]:
# Your code here

Distribution of the weight $\tilde T_i^2$ in the two groups. For a control object the weight equals $\hat e(x)^2$, for a treated object $(1-\hat e(x))^2$: with a propensity around $0.2$ this is roughly $0.04$ against $0.64$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
ax1.hist(e_hat[T[tr] == 0], bins=30, alpha=0.6, color='#3b6ea5', label='control')
ax1.hist(e_hat[T[tr] == 1], bins=30, alpha=0.6, color='#e67e22', label='treated')
ax1.set_xlabel('Propensity ê(x), cross-fitted')
ax1.set_ylabel('Objects')
ax1.set_title('Propensity by group')
ax1.legend()
ax2.hist(w[T[tr] == 0], bins=30, alpha=0.6, color='#3b6ea5', label='control')
ax2.hist(w[T[tr] == 1], bins=30, alpha=0.6, color='#e67e22', label='treated')
ax2.set_xlabel('R-learner weight (T − ê(x))²')
ax2.set_title('Weight by group')
ax2.legend()
fig.tight_layout()
plt.show()

## 3. The R-learner as a weighted regression of the pseudo-outcome

The R-learner loss $\frac1n\sum_i(\tilde Y_i-\tau(X_i)\tilde T_i)^2$ can be rewritten as $\frac1n\sum_i\tilde T_i^2\bigl(\tilde Y_i/\tilde T_i-\tau(X_i)\bigr)^2$. So any learner that accepts `sample_weight` will do: we train it on the features $X$ to predict the pseudo-outcome $\tilde Y_i/\tilde T_i$ with weights $\tilde T_i^2$.

We deliberately take a simple effect model, a linear model with a lasso penalty: the lesson says that $\hat m$ takes over the complexity of the outcome, while a simple, strongly regularized model is enough for $\tau$.

**Task 2. Write a function `r_learner(final)` that trains the model `final` on `X[tr]` with the pseudo-outcome `y_res / t_res` and weights `t_res ** 2` and returns the trained model. Train `Lasso(alpha=0.05)`, store its estimate on the held-out part in `tau_r` and print its PEHE and the number of nonzero coefficients.**

In [ ]:
# Your code here

## 4. Comparison with the schemes of the lesson

Same data, the same outcome model `gb()` inside every scheme: the S-, T-, and X-learner of the lesson “Metalearners: S, T, and X” and the DR-learner, a scheme whose pseudo-outcome is a doubly robust score (the same pseudo-outcome $\varphi_i$ as in the step “From an estimate to a decision: policy learning” of the lesson “How to read the estimate and where to go wrong”). For each scheme we print the PEHE and the error of the average effect on the held-out part.

In [ ]:
logit = lambda: LogisticRegression(max_iter=1000)
est = {
    'S-learner': SLearner(overall_model=gb()).fit(Y[tr], T[tr], X=X[tr]).effect(X[te]),
    'T-learner': TLearner(models=gb()).fit(Y[tr], T[tr], X=X[tr]).effect(X[te]),
    'X-learner': XLearner(models=gb(), propensity_model=logit()).fit(Y[tr], T[tr], X=X[tr]).effect(X[te]),
    'DR-learner': DRLearner(model_regression=gb(), model_propensity=logit(), model_final=gb(),
                            cv=5, random_state=SEED).fit(Y[tr], T[tr], X=X[tr]).effect(X[te]),
    'R-learner (lasso)': tau_r,
}
table = pd.DataFrame({'PEHE ↓': {k: pehe(v) for k, v in est.items()},
                      'error of the average effect': {k: v.mean() - tau[te].mean() for k, v in est.items()}})
print(table.round(3))

True versus estimated effect on the held-out part; the red line is $y=x$. A cloud stretched horizontally means that the scheme barely distinguishes children with large and small effects; a cloud spread vertically means that it is noisy.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8), sharex=True, sharey=True)
lim = [tau[te].min() - 0.5, tau[te].max() + 0.5]
for ax, (name, e) in zip(axes, est.items()):
    ax.scatter(tau[te], e, s=8, alpha=0.5, color='#3b6ea5')
    ax.plot(lim, lim, color='#c0392b', lw=1.2)
    ax.set_title(f'{name}\nPEHE = {pehe(e):.2f}')
    ax.set_xlabel('True effect τ(x)')
axes[0].set_ylabel('Estimated effect τ̂(x)')
axes[0].set_ylim(-2, 10)
fig.tight_layout()
plt.show()

## 5. Sensitivity to the regularization of the effect model

The lesson names the regularization $\Lambda(\tau)$ the main hyperparameter of the R-learner. We check this on two classes of effect model: lasso with penalty weight `alpha` from $10^{-3}$ to $1$, and boosting of depth 1, 2, and 3 (200 trees). The residuals `y_res`, `t_res` stay the same: only the last step changes.

In [ ]:
alphas = np.logspace(-3, 0, 13)
lasso_pehe, lasso_nnz = [], []
for a in alphas:
    m = r_learner(Lasso(alpha=a))
    lasso_pehe.append(pehe(m.predict(X[te])))
    lasso_nnz.append((m.coef_ != 0).sum())
gb_pehe = {d: pehe(r_learner(gb(d)).predict(X[te])) for d in (1, 2, 3)}

print(pd.DataFrame({'alpha': alphas, 'PEHE': lasso_pehe, 'nonzero coefficients': lasso_nnz}).round(3).to_string(index=False))
print('boosting as the effect model:', {f'depth {d}': round(float(v), 3) for d, v in gb_pehe.items()})

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.semilogx(alphas, lasso_pehe, 'o-', color='#3b6ea5', label='R-learner, lasso final model')
for d, v in gb_pehe.items():
    ax.axhline(v, ls='--', lw=1, color=['#e67e22', '#c0392b', '#7f8c8d'][d - 1], label=f'R-learner, boosting depth {d}')
ax.axhline(pehe(est['S-learner']), ls=':', color='black', label='S-learner')
ax.set_xlabel('Lasso penalty alpha')
ax.set_ylabel('PEHE on held-out part ↓')
ax.set_title('R-learner: error versus regularization of τ')
ax.legend(fontsize=8)
plt.show()

**Task 3. From the output of the cell above: at which `alpha` of the grid is the lasso PEHE smallest, and how many nonzero coefficients does the effect model keep at that value? Why does depth-3 boosting err more as the effect model than the same model does in the role of $\hat m$?**

**Your answer:**

## 6. Where the scheme breaks

### 6.1. The propensity model

The pseudo-outcome is divided by $\tilde T_i=T_i-\hat e(X_i)$. If the propensity model confidently predicts $\hat e\approx1$ for a treated object or $\hat e\approx0$ for a control one, the residual is close to zero and the pseudo-outcome is huge. Logistic regression gives smooth probabilities; a default boosting classifier fits harder. We compare the R-learner with the two propensity models and the same outcome model.

In [ ]:
e_gb = cross_val_predict(GradientBoostingClassifier(random_state=SEED), X[tr], T[tr], cv=kf, method='predict_proba')[:, 1]
t_res_gb = T[tr] - e_gb
pseudo = {'logistic regression': y_res / t_res, 'boosting classifier': y_res / t_res_gb}
for name, ps in pseudo.items():
    print(f'{name:>20}: max |pseudo-outcome| = {np.abs(ps).max():8.1f}')

rows = {}
for name, tr_res in (('logistic regression', t_res), ('boosting classifier', t_res_gb)):
    for fname, final in (('lasso, alpha=0.05', Lasso(alpha=0.05)), ('boosting depth 1', gb(1))):
        m = final.fit(X[tr], y_res / tr_res, sample_weight=tr_res ** 2)
        rows[(name, fname)] = pehe(m.predict(X[te]))
print(pd.Series(rows).unstack().round(3))

### 6.2. Group imbalance

The lesson predicts: with comparable groups the orthogonalization of the R-learner wins, under strong imbalance the mixing of the X-learner wins. IHDP has a single treated share, so we take synthetic data in the spirit of Nie and Wager: a complex outcome, a simple effect, and a treated share that we set ourselves:

- Six features, $X\sim U[0,1]^6$.
- Outcome without treatment: $\mu_0(x)=4\sin(\pi x_0x_1)+16(x_2-0.5)^2+2x_3+x_4$.
- The effect is simple and linear: $\tau(x)=(x_0+x_1)/2$.
- A random share $p$ of objects is treated, $Y=\mu_0(X)+T\tau(X)+\varepsilon$, $\varepsilon\sim\mathcal N(0,0.5^2)$.

For each $p\in\{0.5,\ 0.2,\ 0.1,\ 0.05\}$ and three seeds (0, 1, 2) we generate a training sample of 2000 objects and a test sample of 2000 and compare the PEHE of three estimates: the R-learner with lasso, the R-learner with depth-2 boosting, and the X-learner.

**Task 4. Complete the function `imbalance_run(p, seed)`: it generates both samples with `simulate`, computes the Robinson residuals on the training sample (cross-fitting as in task 1), trains the R-learner with `Lasso(alpha=0.01)` and with `gb(2)`, the X-learner with `gb()` and `LogisticRegression(max_iter=1000)`, and returns a dictionary of PEHE values on the test sample. Build the table of means over the three seeds.**

In [ ]:
def simulate(n, p, rng):
    Xs = rng.uniform(0, 1, (n, 6))
    mu0 = 4 * np.sin(np.pi * Xs[:, 0] * Xs[:, 1]) + 16 * (Xs[:, 2] - 0.5) ** 2 + 2 * Xs[:, 3] + Xs[:, 4]
    ts = (Xs[:, 0] + Xs[:, 1]) / 2
    Ts = rng.binomial(1, p, n)
    return Xs, Ts, mu0 + Ts * ts + rng.normal(0, 0.5, n), ts

In [ ]:
# Your code here

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
for (name, col), color in zip(imb.items(), ['#3b6ea5', '#c0392b', '#e67e22']):
    ax.plot(imb.index, col, 'o-', color=color, label=name)
ax.set_xscale('log')
ax.set_xticks(shares)
ax.set_xticklabels([str(s) for s in shares])
ax.invert_xaxis()
ax.set_xlabel('Treated share p (right = stronger imbalance)')
ax.set_ylabel('PEHE, mean of 3 seeds ↓')
ax.set_title('Synthetic data: complex outcome, linear effect')
ax.legend()
plt.show()

**Task 5. Compare the rows of the table from task 4. Which of the three estimates has the PEHE growing fastest from $p=0.5$ to $p=0.05$, and do the numbers confirm the lesson's prediction “under strong imbalance the X-learner wins” for both effect models?**

**Your answer:**

## 7. The R-learner loss as a model selection criterion

On real data the PEHE cannot be computed: there is no true $\tau(x)$. But the R-learner loss $\frac1n\sum_i(\tilde Y_i-\hat\tau(X_i)\tilde T_i)^2$ needs only $Y$, $T$, and the residuals, and it can be computed for **any** estimate $\hat\tau$, including the S-, T-, and X-learner. [Nie and Wager](https://arxiv.org/abs/1712.04912) propose choosing the effect model by this loss on held-out data; `econml.score.RScorer` in EconML works this way.

We compute the residuals on the held-out part with a separate cross-fitting (models trained only on that part), then the loss for every estimate of section 4 and for the R-learner with depth-3 boosting. A useful criterion should order the estimates the same way as PEHE.

In [ ]:
kf_te = KFold(5, shuffle=True, random_state=SEED + 1)
y_res_te = Y[te] - cross_val_predict(gb(), X[te], Y[te], cv=kf_te)
t_res_te = T[te] - cross_val_predict(LogisticRegression(max_iter=1000), X[te], T[te], cv=kf_te, method='predict_proba')[:, 1]
r_loss = lambda e: np.mean((y_res_te - e * t_res_te) ** 2)

cand = dict(est)
cand['R-learner (gb depth 3)'] = r_learner(gb(3)).predict(X[te])
sel = pd.DataFrame({'PEHE ↓': {k: pehe(v) for k, v in cand.items()},
                    'R-loss ↓': {k: r_loss(v) for k, v in cand.items()}}).sort_values('PEHE ↓')
print(sel.round(4))
rho = spearmanr(sel['PEHE ↓'], sel['R-loss ↓']).statistic
print(f'Spearman rank correlation of PEHE and R-loss over {len(sel)} estimates: {rho:.3f}')

The R-learner loss is not PEHE even up to a constant: it includes the outcome noise and the errors of the residuals, so it may swap estimates that are close in PEHE: here the X-learner, third by PEHE, comes first by the R-learner loss. The grossest error, the R-learner with depth-3 boosting, it puts last.

## 8. The library call

In EconML, the R-learner with an arbitrary effect model is `econml.dml.NonParamDML`: `model_y` is the model $\hat m$, `model_t` the propensity model, `model_final` the effect model (it must accept `sample_weight`), `cv` the number of cross-fitting folds. In CausalML the same scheme is `causalml.inference.meta.BaseRRegressor`. The folds inside the library differ, so the number will not match ours to the third decimal.

In [ ]:
npd = NonParamDML(model_y=gb(), model_t=LogisticRegression(max_iter=1000), model_final=Lasso(alpha=0.05),
                  discrete_treatment=True, cv=5, random_state=SEED).fit(Y[tr], T[tr], X=X[tr])
print(f'NonParamDML (lasso, alpha=0.05): PEHE {pehe(npd.effect(X[te])):.3f}  | our R-learner: {pehe(tau_r):.3f}')

## What to take away

- **The R-learner is two auxiliary models and one weighted regression.** All of the orthogonalization lives in the Robinson residuals with cross-fitting; the effect model sees only the pseudo-outcome and the weights $\tilde T_i^2$.
- **The regularization of the effect model matters more than the choice of scheme.** On IHDP, lasso with a tuned penalty and depth-3 boosting give CATE estimation errors that differ several-fold with the same residuals.
- **The propensity model is the weak spot.** A confident propensity model makes the residuals $\tilde T_i$ almost zero, the pseudo-outcome explodes, and a flexible effect model fits the outliers.
- **Group imbalance hurts a flexible effect model.** With a small treated share the weight rests on a few objects; a simple effect model tolerates this, boosting does not, and then the X-learner is more reliable.
- **The R-learner loss is a selection criterion without the true effect.** It is not PEHE, but it filters out gross errors and is suitable for tuning hyperparameters on real data.

Sources:
1. [Nie and Wager, 2021. Quasi-oracle estimation of heterogeneous treatment effects](https://arxiv.org/abs/1712.04912).
2. [Hill, 2011. Bayesian nonparametric modeling for causal inference](https://doi.org/10.1198/jcgs.2010.08162).
3. [Künzel et al., 2019. Metalearners for estimating heterogeneous treatment effects using machine learning](https://arxiv.org/abs/1706.03461).
4. [EconML](https://github.com/py-why/EconML), [CausalML](https://github.com/uber/causalml).
5. [Louizos, Shalit, Mooij, and Sontag, 2017. Causal effect inference with deep latent-variable models](https://arxiv.org/abs/1705.08821); the IHDP data — the first of ten replications in the [CEVAE repository](https://github.com/AMLab-Amsterdam/CEVAE); the file `ihdp_npci_1.csv` in the `data` folder of the course repository is a copy from [uber/causalml](https://github.com/uber/causalml) (Apache-2.0 license), byte for byte identical to the CEVAE file.